# From a flat table to BHoM tables: silver, a mapping, and gold

Data engineers can prototype with **flat tables** and still end up with tables that follow the BHoM schema. This notebook shows the flow:

- **Bronze**: the Revit data as extracted, in Parquet files in a Unity Catalog volume. Read-only.
- **Silver**: a big flat table that you make in your notebook from the Parquet files, with as many columns as you like. No BHoM schema is enforced.
- **A mapping file** that says which silver column is which property of which BHoM class, in which unit, with which enum values. Genie Code helps you create and maintain it.
- **Gold**: BHoM tables, one per class, written from the silver table and its mapping. Every row is checked by the BHoM classes.

**What you will see**

1. Install or update the package
2. Settings
3. Bronze to silver: a flat table
4. Look at the table and at the BHoM classes
5. A mapping, checked: the problems and the fixes
6. Write the gold tables, and read them back as Python objects
7. The table changes: the alignment check
8. The same with Genie Code
9. Clean up

**To run it:** attach a cluster or serverless compute with Python 3.12 or later, set the catalog and the two schemas in section 2 (the schemas must exist and you must be allowed to create tables in them), and use *Run all*.
Nothing in this notebook is expected to fail. The silver table is named `bhom_demo_elements_flat`, the gold tables are `Building`, `Storey` and `Beam`, and section 9 can drop them.

## 1. Install or update the package

`bhom-schema[mapping]` is the package with the mapping tools (the extra adds PyYAML, which Databricks runtimes already have). `--upgrade --pre` fetches the latest build even if one is already installed, and the
Python process is restarted afterwards so the new version is the one that is imported. `%pip` and the restart must each be in a cell of their own.

In [ ]:
%pip install --upgrade --pre --no-cache-dir --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "bhom-schema[mapping]"

In [ ]:
dbutils.library.restartPython()

In [ ]:
import bhom_schema

print("package version :", bhom_schema.__version__)
print("BHoM version    :", bhom_schema.__bhom_version__)

## 2. Settings

The tables go in two schemas of one catalog: the flat silver table in `schema`, and the BHoM gold tables in `gold_schema`. **Change the widgets at the top of the notebook to a catalog and schemas of your own**
(both schemas must already exist and you must be allowed to create tables in them; the defaults are only an example).

The mapping file is kept in a `bhom-mappings` folder next to the notebooks of your tool, and found again from the table name, from this folder or from any folder above it. Here that folder is the folder of this notebook.

In [ ]:
import os

dbutils.widgets.text("catalog", "student_area", "Catalog")
dbutils.widgets.text("schema", "vcl", "Schema of the silver table")
dbutils.widgets.text("gold_schema", "vcl_gold", "Schema of the gold tables")
dbutils.widgets.dropdown("cleanup", "no", ["no", "yes"], "Drop the demo tables at the end")

CATALOG = dbutils.widgets.get("catalog")
SILVER_SCHEMA = dbutils.widgets.get("schema")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")
SILVER = f"{CATALOG}.{SILVER_SCHEMA}.bhom_demo_elements_flat"
GOLD = f"{CATALOG}.{GOLD_SCHEMA}"
MAPPING_FOLDER = os.getcwd()

# Fails here, with a clear message, when a schema does not exist or cannot be read
for schema_name in (SILVER_SCHEMA, GOLD_SCHEMA):
    spark.sql(f"DESCRIBE SCHEMA `{CATALOG}`.`{schema_name}`")

print("silver table  :", SILVER)
print("gold schema   :", GOLD)
print("mapping folder:", MAPPING_FOLDER)

## 3. Bronze to silver: a flat table

In your own work this is where you read the Parquet files of the bronze volume (`spark.read.parquet("/Volumes/<catalog>/<schema>/<volume>/<project>/<model>/elements.parquet")`) and flatten them into one table.
Here a small sample stands in, so the notebook runs anywhere. It is deliberately like real data: the project and model values repeat on every row, storey heights are in **millimetres**, materials are **free text** (one with a typo),
one column (`qa_flag`) has no BHoM home, and one row disagrees with the others about the number of levels of its building. A `source_file` column keeps the lineage back to the Parquet file.

**Expect:** twelve rows written to the silver table.

In [ ]:
from bhom_schema.tables.sql import quote_table

SILVER_DDL = (
    "`project_number` STRING, `model_name` STRING, `levels_above_ground` INT, `typical_level_gap` DOUBLE, "
    "`structural_material` STRING, `family_type` STRING, `source_file` STRING, `qa_flag` STRING")

def sample_rows():
    """Returns the rows of the sample silver table: two projects with two models each and three beam families per model."""
    rows = []
    for project in ("P-1", "P-2"):
        for model, levels, height in (("Tower", 24, 3900.0), ("Podium", 4, 4200.0)):
            for index, family in enumerate(("UB-0", "UB-1", "UB-2")):
                rows.append({
                    "project_number": project, "model_name": model, "levels_above_ground": levels + (1 if project == "P-2" else 0), "typical_level_gap": height,
                    "structural_material": "glulam" if index == 1 else "Steel - hot rolled", "family_type": family,
                    "source_file": f"{project}/{model}/{family}.parquet", "qa_flag": "ok"})

    return rows

SILVER_ROWS = sample_rows()
# A typo in the material of one row
SILVER_ROWS[11]["structural_material"] = "Steel hot roled"
# One row of the first tower says it has 25 levels, the others say 24
SILVER_ROWS[1].update({"levels_above_ground": 25, "qa_flag": "check"})

def write_silver(rows, ddl):
    """Writes the rows as the silver table, replacing what was there."""
    spark.createDataFrame(rows, schema=ddl).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(quote_table(SILVER))

write_silver(SILVER_ROWS, SILVER_DDL)
print(f"wrote {len(SILVER_ROWS)} rows to {SILVER}")

## 4. Look at the table and at the BHoM classes

These are the **facts** that you (or Genie Code) work from. None of them decides anything: which column means what is a judgment, and it is yours or the agent's.

`alignment` answers the question "is this table BHoM-aligned?" in a few lines. `profile` says what each column holds, `sample_rows` shows a few real rows, and `check_key` tests a key that you propose (here: a building is identified by its
project and its model). **Expect:** no mapping yet; the profile of the eight columns; two rows; and the key made of 4 groups, with the column that is constant in them (the storey height) and the columns that vary within a building. `levels_above_ground` is among
those that vary, and it should not be: one row disagrees with the others of its building, which section 5 finds.

In [ ]:
from bhom_schema.mapping import alignment, check_key, profile, sample_rows

print(alignment(spark, SILVER).summary())
print()
print(profile(spark, SILVER).text())
print()
for row in sample_rows(spark, SILVER, columns=["model_name", "typical_level_gap", "structural_material"], n=2):
    print(row)

print()
print(check_key(spark, SILVER, ["project_number", "model_name"]).text())

The BHoM classes are read from the schema itself. `schema_overview` lists them, `describe_class` gives the card of one (descriptions, aliases, units, the members of every enum, and the fields that need a decision), and `search_properties`
finds properties by words (it filters; it does not rank). **Expect:** the classes of the DataSchema assembly, the card of `Storey`, and the properties found for "storeys above ground".

In [ ]:
from bhom_schema.mapping import describe_class, schema_overview, search_properties

for item in schema_overview():
    print(f"{item.name:26} {item.properties} properties   {item.summary[:70]}")

print()
print(describe_class("Storey").text())
print()
for match in search_properties("storeys above ground"):
    print(match.label, "(found in", ", ".join(match.found_in) + ")")

## 5. A mapping, checked: the problems and the fixes

A mapping file says, for each BHoM class the table holds rows of (an *entity*): the silver columns that identify one object (its `key`), its `parent`, and for each property the silver column it comes from, its unit and the text-to-member map of an enum.
Columns with no BHoM home are listed under `unmapped`, with the reason. The file is plain YAML that you can read and edit; `load_mapping` checks every rule when it is read (an unknown class, property, unit or enum member is refused),
and `validate_mapping` checks it against the rows.

Here the mapping is written as a Python dictionary and saved. **Expect:** the path of the file, and then its content.

In [ ]:
import copy

from bhom_schema.mapping import load_mapping, parse_mapping, save_mapping

BASE = {"Fragments": "skip", "CustomData": "skip"}
MAPPING_DATA = {
    "version": 1,
    "source": SILVER,
    "gold_schema": GOLD,
    "entities": {
        "Building": {
            "class": "bhom_schema.DataSchema_oM.Building.Building",
            "key": ["project_number", "model_name"],
            "fields": dict(BASE),
            "columns": {"Name": {"from": "model_name"}, "StoreysAboveGround": {"from": "levels_above_ground"}},
        },
        "Storey": {
            "class": "bhom_schema.DataSchema_oM.Storey.Storey",
            "key": ["project_number", "model_name"],
            "parent": "Building",
            "fields": dict(BASE),
            "columns": {"TypicalHeight": {"from": "typical_level_gap", "unit": "mm"}},
        },
        "Beam": {
            "class": "bhom_schema.DataSchema_oM.Elements.Beam.Beam",
            "key": ["project_number", "model_name", "family_type"],
            "parent": "Building",
            "fields": {**BASE, "MaterialDistribution": "json"},
            "columns": {
                "TypicalMaterial": {"from": "structural_material", "values": {"Steel - hot rolled": "SteelHotRolled", "glulam": "TimberGlulam"}},
                "TypicalSize": {"from": "family_type"},
            },
        },
    },
    "unmapped": {"source_file": "lineage back to the bronze Parquet file; stays in silver", "qa_flag": "internal check of this tool; stays in silver"},
}

path = save_mapping(parse_mapping(MAPPING_DATA), folder=MAPPING_FOLDER)
print(path)
print()
print(path.read_text(encoding="utf-8"))

Now the check. `validate_mapping` applies the mapping to the rows, as `write_gold` would, and reports everything that does not fit. **Expect:** two problems with their counts and examples: one beam material that is not an enum member
and not in the value map (the typo), and the building whose rows disagree about its levels.

In [ ]:
from bhom_schema.mapping import validate_mapping

report = validate_mapping(spark, SILVER)
print(report.summary())

Each problem has its own fix. The typo is fixed in the mapping, by adding the text to the value map. The levels that disagree are fixed where they come from: the silver table. (Genie Code would propose the first as a change to the mapping file, and you would approve it.)
**Expect:** the mapping saved again, and a valid report.

In [ ]:
MAPPING_DATA["entities"]["Beam"]["columns"]["TypicalMaterial"]["values"]["Steel hot roled"] = "SteelHotRolled"
save_mapping(parse_mapping(MAPPING_DATA), folder=MAPPING_FOLDER)

SILVER_ROWS[1].update({"levels_above_ground": 24, "qa_flag": "ok"})
write_silver(SILVER_ROWS, SILVER_DDL)

print(validate_mapping(spark, SILVER).summary())

## 6. Write the gold tables, and read them back as Python objects

`write_gold` makes one table per entity in the gold schema (`Building`, `Storey`, `Beam`), with the columns, units and comments of the BHoM classes and Delta constraints on the enum columns. A storey or a beam gets a `Building_id` column with the id of its
building, so the tables stay linked. It reads **all** the rows and refuses to write anything while the mapping is not valid. Ids are made from the entity and its key values, so writing again does not add the same row twice.
**Expect:** the three tables with their row counts, and one row of the beam table.

In [ ]:
from bhom_schema.mapping import write_gold

MAPPING = load_mapping(SILVER)
result = write_gold(spark, SILVER, MAPPING, replace=True)
print(result.text())
print()
display(spark.table(quote_table(result.tables["Beam"])))

The gold rows are BHoM objects: `read_objects` gives them back with enums as Python enum members and numbers in the unit of the property (the heights were in millimetres in silver, they are metres here). The children are linked to their building
through `Building_id`. **Expect:** the height of each building's storey in metres, and for each beam its size, its material and the building it belongs to.

In [ ]:
from bhom_schema.DataSchema_oM.Building import Building
from bhom_schema.DataSchema_oM.Elements.Beam import Beam
from bhom_schema.DataSchema_oM.Storey import Storey
from bhom_schema.tables.spark import read_objects, read_rows

buildings = read_objects(spark, Building, result.tables["Building"], fields=MAPPING.entity("Building").fields, keyed=True)
storeys = read_objects(spark, Storey, result.tables["Storey"], fields=MAPPING.entity("Storey").fields, keyed=True)

# The storey rows say which building they belong to; the objects are matched by the id of the row
for row in read_rows(spark, result.tables["Storey"]):
    print(f"{buildings[row['Building_id']].Name:8} storey height {storeys[row['BHoM_Guid']].TypicalHeight:.2f} m")

print()
beams = read_objects(spark, Beam, result.tables["Beam"], fields=MAPPING.entity("Beam").fields, keyed=True)
for row in read_rows(spark, result.tables["Beam"], limit=4):
    beam = beams[row["BHoM_Guid"]]
    print(f"{beam.TypicalSize:6} {beam.TypicalMaterial.name:15} in {buildings[row['Building_id']].Name}")

## 7. The table changes: the alignment check

Silver tables change: columns are added while you work. `alignment` is the quick check to run now and then (or to ask Genie Code for). Here a column with the number of basement levels is added to the silver table. **Expect:** the mapping no longer matches the
table (a new column it does not know), the same finding from `validate_mapping`, and after the column is mapped, aligned again.

In [ ]:
NEW_DDL = SILVER_DDL + ", `levels_below_ground` INT"
for row in SILVER_ROWS:
    row["levels_below_ground"] = 2 if row["model_name"] == "Tower" else None

write_silver(SILVER_ROWS, NEW_DDL)
print(alignment(spark, SILVER).summary())
print()
print(validate_mapping(spark, SILVER, limit=100).summary())

In [ ]:
MAPPING_DATA["entities"]["Building"]["columns"]["StoreysBelowGround"] = {"from": "levels_below_ground", "default": 0}
save_mapping(parse_mapping(MAPPING_DATA), folder=MAPPING_FOLDER)

print(alignment(spark, SILVER).summary())

## 8. The same with Genie Code

You do not have to write any of this by hand. The admins of your workspace publish the `bhom-mapping` skill, and Genie Code (the panel next to your notebook) then does the work: it looks at the table and the classes with the tools of section 4, proposes the mapping,
checks it with `validate_mapping`, and explains the problems. You approve every change to the mapping file, and every write to gold.

Try these in the Genie Code panel, in this notebook, in this order (the answers vary, so they cannot be stored here):

1. `@bhom-mapping check <catalog>.<schema>.bhom_demo_elements_flat` : it runs the alignment check and tells you in a few lines whether the table is aligned and what is missing.
2. Delete the file `bhom-mappings/<table>.yml` first, then ask `@bhom-mapping map <table>` : it profiles the table, proposes the entities, tests the keys with `check_key`, maps the columns in batches and asks you when the data does not settle a choice
   (for example whether `typical_level_gap` is in millimetres), shows you the mapping file as a diff, and waits for your approval.
3. Compare what it proposed with the mapping of section 5. Differences are fine when yours are right; tell it what it got wrong and it will adjust.
4. `@bhom-mapping write gold <table>` : it validates all rows and writes the gold tables, only if you ask.

If Genie Code does not know `@bhom-mapping`, the skill has not been published to your workspace yet: ask an admin (see the notes of the plan for admins).

## 9. Clean up

Set the `cleanup` widget to `yes` (and run this cell) to drop the silver table and the gold tables this notebook created, and the mapping file. The default is `no`, so you can look at the tables in the catalog explorer first.

In [ ]:
names = [SILVER] + [f"{GOLD}.{entity}" for entity in ("Building", "Storey", "Beam")]

if dbutils.widgets.get("cleanup") == "yes":
    for name in names:
        spark.sql(f"DROP TABLE IF EXISTS {quote_table(name)}")
        print("dropped", name)

    mapping_file = path
    if mapping_file.exists():
        mapping_file.unlink()
        print("removed", mapping_file)
else:
    print("kept these tables and the mapping file (set the cleanup widget to yes to drop them):")
    for name in names:
        print("  ", name)

    print("  ", path)

## Summary

You made a flat silver table, looked at it and at the BHoM classes with tools that report facts, wrote and checked a mapping, wrote the gold tables as BHoM tables, read them back as Python objects, and saw how the alignment check follows a table that changes.
With Genie Code in the panel next to your notebook, the same work is a conversation. The notebook `Databricks-DataSchema-Demo.ipynb` shows the BHoM tables themselves in more detail.